## Plot PPA choice evolution under calibrated mutation

Reads PPA simulation outputs from:

```text
Code_Submission/simulation_mutation/Output files (Risk Neutral, Mutation, Verified)
```

The notebook supports both the original latent-delta scenarios and the calibrated scenarios generated by `mutation_sample_generation.ipynb`. When `target_physical_shift` is present, panel labels and optional filters use the calibrated final physical Pearson shift rather than the latent delta.

## 0. Imports


In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Dict, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D


In [ ]:
# ============================================================
# Submission dtype helpers
# ============================================================
# These wrappers reduce DataFrame memory use without changing the financial
# calculations: identifiers/counters are downcast, repeated labels become
# categoricals, and continuous numerical columns remain float64.
import numpy as np

_PD_READ_CSV = pd.read_csv
_PD_READ_EXCEL = pd.read_excel

_INTEGER_DTYPE_CANDIDATES = {
    "match_id": np.int32,
    "hour": np.int16,
    "hour_index": np.int16,
    "replication": np.int16,
    "case_order": np.int16,
    "enabled": np.int8,
    "rank": np.int32,
}

_CATEGORY_DTYPE_CANDIDATES = {
    "case_id",
    "case_family",
    "case_label",
    "combined_category",
    "metric",
    "mutation_axis",
    "mutation_direction",
    "mutation_family",
    "mutation_label",
    "ppa_type",
    "profile_type",
    "risk_group",
    "risk_label",
    "scenario_name",
    "scenario_type",
    "solution_type",
    "status",
    "variable",
    "var_i",
    "var_j",
}


def _integer_dtype_fits(values, dtype) -> bool:
    if len(values) == 0:
        return True
    info = np.iinfo(dtype)
    return float(np.nanmin(values)) >= info.min and float(np.nanmax(values)) <= info.max


def optimize_dataframe_dtypes(df: pd.DataFrame) -> pd.DataFrame:
    """Conservatively compact non-financial columns after file loading."""
    if not isinstance(df, pd.DataFrame) or df.empty:
        return df

    for col in df.columns:
        series = df[col]
        if pd.api.types.is_integer_dtype(series.dtype):
            df[col] = pd.to_numeric(series, downcast="integer")

    for col, dtype in _INTEGER_DTYPE_CANDIDATES.items():
        if col not in df.columns:
            continue
        numeric = pd.to_numeric(df[col], errors="coerce")
        if numeric.isna().any():
            continue
        values = numeric.to_numpy(dtype="float64", copy=False)
        rounded = np.rint(values)
        if np.array_equal(values, rounded) and _integer_dtype_fits(rounded, dtype):
            df[col] = rounded.astype(dtype, copy=False)

    n_rows = len(df)
    for col in _CATEGORY_DTYPE_CANDIDATES.intersection(df.columns):
        series = df[col]
        if pd.api.types.is_categorical_dtype(series.dtype):
            continue
        if not (pd.api.types.is_object_dtype(series.dtype) or pd.api.types.is_string_dtype(series.dtype)):
            continue
        non_null = series.dropna()
        if non_null.empty:
            continue
        n_unique = int(non_null.nunique())
        if n_unique <= min(128, max(2, n_rows // 2)):
            df[col] = series.astype("category")

    return df


def read_csv_optimized(*args, **kwargs) -> pd.DataFrame:
    return optimize_dataframe_dtypes(_PD_READ_CSV(*args, **kwargs))


def read_excel_optimized(*args, **kwargs) -> pd.DataFrame:
    return optimize_dataframe_dtypes(_PD_READ_EXCEL(*args, **kwargs))


## 1. Control on plot content

Edit this section to decide **what observations appear in the plot**. Leave a filter as `None` to keep all available values.


In [ ]:
# ============================================================
# SECTION 1. CONTROL ON PLOT CONTENT
# ============================================================

# -----------------------------
# Main input/output locations
# -----------------------------
# Default local structure:
#   Code_Submission/
#     simulation_mutation/
#       plot_mutation_choice_evolution.ipynb
#       Output files (Risk Neutral, Mutation, Verified)/
#
# VS Code/Jupyter may set Path.cwd() to Code_Submission rather than to the notebook folder.
# Use the simulation_mutation output folder explicitly to avoid reading stale outputs under Code_Submission/.
results_dir = str(Path("simulation_mutation") / "Output files (Risk Neutral, Mutation, Verified)")
target_plot_data_file = None  # Set to a CSV path to override auto-detection.
save_dir_name = "Figures and plot tables"

# Compact exported choice tables.
# The CSV contains only the scenario/match-level PPA choice variables used by the plot.
# The XLSX workbook contains that same table plus a scenario-level choice-count sheet.
plot_export_decimal_places = 2
export_choice_summary_workbook = True
plot_export_drop_columns = ["sample_file", "sample_source", "sample_path_rel", "code_root"]  # retained only for backward compatibility

# Optional override. Usually leave this as None.
# Example:
# code_root_override = "/absolute/path/to/Code for submission"
code_root_override = None
simulation_subfolder = "simulation_mutation"
prefer_simulation_mutation_folder = True

# -----------------------------
# Mutation/scenario controls
# -----------------------------
# Which mutation family to plot:
#   "shape", "basis", "load_price", "cannibalization"
mutation_family_to_plot = "cannibalization"

# Keep the no-mutation baseline panel beside the selected mutation-family panels.
include_baseline_panel = True

# Optional mutation-level filters. These are applied to mutation scenarios only;
# the baseline panel is retained when include_baseline_panel = True.
filter_mutation_level_labels = None   # e.g. ["low", "medium", "high"] or None
filter_mutation_target_deltas = None  # latent deltas; e.g. [-0.30, -0.50] or None
filter_target_physical_shifts = None  # calibrated final physical shifts; e.g. [-0.10, -0.30, -0.50] or None
filter_scenario_names = None          # e.g. ["Mutation__ShapeDeterioration__target_physical_shift_m0p30"] or None

# -----------------------------
# Match/profile/content filters
# -----------------------------
# Show only matches that started from one baseline selected profile.
# Useful values: "Fix", "AsC", "AsG", "No Contract", or None.
baseline_profile_filter = None

# Current selected profile after the mutation/baseline scenario.
# Useful values: "Fix", "AsC", "AsG", "No Contract".
filter_current_profiles = None  # e.g. ["Fix", "AsC"] or None

# PPA type marker filter.
# Useful values: "Physical", "Virtual", "No Contract".
filter_ppa_types = None  # e.g. ["Physical", "No Contract"] or None

# Match IDs to keep. This is the most direct way to inspect specific matches.
filter_match_ids = None  # e.g. [1, 2, 3] or None

# -----------------------------
# Axis data controls
# -----------------------------
# Axis source:
#   "simulated" is recommended for mutation because it shows realized mutated values.
#   "original" uses historical match files.
#   "ref" uses the simulation notebook's original-preferred fallback.
correlation_source = "simulated"
level_source = "simulated"

# Which scatter map to draw:
#   "correlation" -> x = profile-shape correlation, y = nodal-price correlation
#   "mean"        -> x = mean volume mismatch index, y = mean price spread index
#   "median"      -> x = median volume mismatch index, y = median price spread index
scatter_map = "correlation"

# -----------------------------
# Output controls
# -----------------------------
save_fig = True
export_filtered_plot_table = True
plot_choice_evolution_figure = True
plot_switch_bars_figure = True

choice_fig_name_template = "Mutation_Choice_Evolution__{mutation_family}__{scatter_map}.png"
switch_fig_name_template = "Mutation_Profile_Switch_Bars__{mutation_family}.png"
filtered_plot_table_name_template = "Filtered_Mutation_PPA_Choices__{mutation_family}.csv"
choice_summary_workbook_name_template = "Filtered_Mutation_PPA_Choices__{mutation_family}.xlsx"

CONTENT_CONTROLS = {
    "results_dir": results_dir,
    "target_plot_data_file": target_plot_data_file,
    "save_dir_name": save_dir_name,
    "plot_export_decimal_places": plot_export_decimal_places,
    "export_choice_summary_workbook": export_choice_summary_workbook,
    "plot_export_drop_columns": plot_export_drop_columns,
    "code_root_override": code_root_override,
    "simulation_subfolder": simulation_subfolder,
    "prefer_simulation_mutation_folder": prefer_simulation_mutation_folder,
    "mutation_family_to_plot": mutation_family_to_plot,
    "include_baseline_panel": include_baseline_panel,
    "filter_mutation_level_labels": filter_mutation_level_labels,
    "filter_mutation_target_deltas": filter_mutation_target_deltas,
    "filter_target_physical_shifts": filter_target_physical_shifts,
    "filter_scenario_names": filter_scenario_names,
    "baseline_profile_filter": baseline_profile_filter,
    "filter_current_profiles": filter_current_profiles,
    "filter_ppa_types": filter_ppa_types,
    "filter_match_ids": filter_match_ids,
    "correlation_source": correlation_source,
    "level_source": level_source,
    "scatter_map": scatter_map,
    "save_fig": save_fig,
    "export_filtered_plot_table": export_filtered_plot_table,
    "plot_choice_evolution_figure": plot_choice_evolution_figure,
    "plot_switch_bars_figure": plot_switch_bars_figure,
    "choice_fig_name_template": choice_fig_name_template,
    "switch_fig_name_template": switch_fig_name_template,
    "filtered_plot_table_name_template": filtered_plot_table_name_template,
    "choice_summary_workbook_name_template": choice_summary_workbook_name_template,
}


## 2. Control on plotting format

Edit this section to control **how the plot looks**. These settings do not change which observations are included.


In [ ]:
# ============================================================
# SECTION 2. CONTROL ON PLOTTING FORMAT
# ============================================================

# -----------------------------
# Figure layout and markers
# -----------------------------
figsize = (14, 11)
point_size = 105
point_alpha = 0.90
edge_color = "black"
edge_width = 0.6

# Match-ID labels
show_match_id_labels = False
label_font_size = 7
label_alpha = 0.80

# -----------------------------
# Font sizes
# -----------------------------
tick_font_size = 14
axis_title_font_size = 15
subplot_title_font_size = 14
legend_font_size = 14
legend_title_font_size = 14
legend_marker_size = 11

# -----------------------------
# Axis limits and reference lines
# -----------------------------
# Leave as None to use common data-driven limits across all panels.
x_axis_lim = None  # e.g. (-1.0, 1.0)
y_axis_lim = None  # e.g. (-1.0, 1.0)
axis_zero_line_width = 1.0
axis_limit_padding_fraction = 0.06

# -----------------------------
# Jitter controls
# -----------------------------
# Use jitter only when points overlap heavily.
use_jitter = False
jitter_scale_x = 0.003
jitter_scale_y = 0.003
random_seed = 42

# -----------------------------
# Marker/color grammar
# -----------------------------
# Marker = PPA type.
marker_map = {
    "No Contract": "X",
    "Physical": "o",
    "Virtual": "^",
}

# Color = selected profile type.
color_map = {
    "No Contract": "#7f7f7f",
    "Fix": "#1f77b4",
    "AsG": "#ff7f0e",
    "AsC": "#2ca02c",
    "Unknown": "#d62728",
}

# Legend settings for the choice-evolution panels.
# PPA-type legend is removed by default because the current request is to show only
# the profile-type legend in each subfigure.
show_ppa_type_legend = False
show_profile_type_legend = True
profile_legend_per_panel = True
profile_legend_loc = "best"
profile_legend_count_label = True

# -----------------------------
# Switch-bar figure format
# -----------------------------
bar_figsize_min_width = 6
bar_figsize_per_profile = 5
bar_figsize_height = 5
bar_ylim = (0, 1.02)
bar_xlabel = "$\\Delta\\rho$"
bar_ylabel = "Share"
bar_legend_title = "Mutated profile"

PLOT_FORMAT_CONTROLS = {
    "figsize": figsize,
    "point_size": point_size,
    "point_alpha": point_alpha,
    "edge_color": edge_color,
    "edge_width": edge_width,
    "show_match_id_labels": show_match_id_labels,
    "label_font_size": label_font_size,
    "label_alpha": label_alpha,
    "tick_font_size": tick_font_size,
    "axis_title_font_size": axis_title_font_size,
    "subplot_title_font_size": subplot_title_font_size,
    "legend_font_size": legend_font_size,
    "legend_title_font_size": legend_title_font_size,
    "legend_marker_size": legend_marker_size,
    "x_axis_lim": x_axis_lim,
    "y_axis_lim": y_axis_lim,
    "axis_zero_line_width": axis_zero_line_width,
    "axis_limit_padding_fraction": axis_limit_padding_fraction,
    "use_jitter": use_jitter,
    "jitter_scale_x": jitter_scale_x,
    "jitter_scale_y": jitter_scale_y,
    "random_seed": random_seed,
    "marker_map": marker_map,
    "color_map": color_map,
    "show_ppa_type_legend": show_ppa_type_legend,
    "show_profile_type_legend": show_profile_type_legend,
    "profile_legend_per_panel": profile_legend_per_panel,
    "profile_legend_loc": profile_legend_loc,
    "profile_legend_count_label": profile_legend_count_label,
    "bar_figsize_min_width": bar_figsize_min_width,
    "bar_figsize_per_profile": bar_figsize_per_profile,
    "bar_figsize_height": bar_figsize_height,
    "bar_ylim": bar_ylim,
    "bar_xlabel": bar_xlabel,
    "bar_ylabel": bar_ylabel,
    "bar_legend_title": bar_legend_title,
}


## 3. Assemble settings

This cell merges the content controls and format controls into one flat settings dictionary used by the functions below.


In [ ]:
SETTINGS = {
    **CONTENT_CONTROLS,
    **PLOT_FORMAT_CONTROLS,
}

print("Settings assembled.")
print("  Mutation family:", SETTINGS["mutation_family_to_plot"])
print("  Scatter map    :", SETTINGS["scatter_map"])


## 4. Path helpers


In [ ]:
NOTEBOOK_CWD = Path.cwd().resolve()


def _as_path_or_none(value) -> Optional[Path]:
    if value is None:
        return None
    text = str(value).strip()
    if text == "":
        return None
    return Path(text).expanduser()


def _dedupe_paths(paths) -> list[Path]:
    out = []
    seen = set()
    for path in paths:
        if path is None:
            continue
        p = Path(path).expanduser()
        try:
            key = str(p.resolve())
        except Exception:
            key = str(p)
        if key not in seen:
            seen.add(key)
            out.append(p)
    return out


def _project_root_candidates(settings: Dict, max_parent_depth: int = 6) -> list[Path]:
    """Candidate Code_Submission roots.

    This is intentionally more defensive than the earlier version:
    - if the kernel cwd is Code_Submission, include Code_Submission and Code_Submission/simulation_mutation
    - if the kernel cwd is Code_Submission/simulation_mutation, include Code_Submission first
    - if an absolute code_root_override is supplied, prioritize it
    """
    sim_subfolder = str(settings.get("simulation_subfolder", "simulation_mutation")).strip() or "simulation_mutation"
    override = _as_path_or_none(settings.get("code_root_override"))

    anchors = []
    if override is not None:
        anchors.append(override)
    anchors.append(NOTEBOOK_CWD)

    # Include parents so running from a child folder still works.
    for parent in list(NOTEBOOK_CWD.parents)[:max_parent_depth]:
        anchors.append(parent)

    roots = []
    for anchor in anchors:
        p = Path(anchor).expanduser()

        # If the anchor itself is simulation_mutation, its parent is the likely Code_Submission root.
        if p.name == sim_subfolder:
            roots.append(p.parent)

        # If the anchor contains simulation_mutation, the anchor is likely the Code_Submission root.
        if (p / sim_subfolder).exists():
            roots.append(p)

        # Also keep the anchor itself as a fallback because some users execute from the output root.
        roots.append(p)

    return [p.resolve() if p.exists() else p for p in _dedupe_paths(roots)]


def _candidate_base_dirs(settings: Dict) -> list[Path]:
    """Candidate directories from which relative input/output paths may be resolved."""
    sim_subfolder = str(settings.get("simulation_subfolder", "simulation_mutation")).strip() or "simulation_mutation"
    prefer_sim = bool(settings.get("prefer_simulation_mutation_folder", True))

    bases = []
    for root in _project_root_candidates(settings):
        sim_dir = root / sim_subfolder
        if prefer_sim:
            bases.extend([sim_dir, root])
        else:
            bases.extend([root, sim_dir])

    # Keep the current working directory in the list, but not necessarily first.
    bases.append(NOTEBOOK_CWD)

    return [p.resolve() if p.exists() else p for p in _dedupe_paths(bases)]


def _candidate_results_dirs(settings: Dict) -> list[Path]:
    results_candidate = Path(str(settings["results_dir"])).expanduser()

    if results_candidate.is_absolute():
        return _dedupe_paths([results_candidate])

    dirs = []
    for base in _candidate_base_dirs(settings):
        dirs.append(base / results_candidate)

    # Also support users who set results_dir to an already-relative path such as
    # "simulation_mutation/Output files (...)".
    for root in _project_root_candidates(settings):
        dirs.append(root / results_candidate)

    # Common fallback: any immediate "Output files*" folder under the candidate bases.
    for base in _candidate_base_dirs(settings):
        if base.exists() and base.is_dir():
            try:
                dirs.extend(sorted(base.glob("Output files*")))
            except Exception:
                pass

    return [p.resolve() if p.exists() else p for p in _dedupe_paths(dirs)]


def first_existing(*paths) -> Optional[Path]:
    for path in paths:
        if path is None:
            continue
        p = Path(path).expanduser()
        if p.exists():
            return p.resolve()
    return None


def find_plot_data_candidates(settings: Dict) -> list[Path]:
    """Return existing candidate Simulation_Plot_Data_All_Matches.csv paths in priority order."""
    target = _as_path_or_none(settings.get("target_plot_data_file"))

    if target is not None:
        candidates = [target]

        # If target is relative, try it under results dirs and base dirs.
        if not target.is_absolute():
            candidates.extend([d / target for d in _candidate_results_dirs(settings)])
            candidates.extend([b / target for b in _candidate_base_dirs(settings)])

        return [p.resolve() for p in _dedupe_paths(candidates) if p.exists()]

    filename = "Simulation_Plot_Data_All_Matches.csv"
    candidates = []

    # Primary expected location: output folder.
    candidates.extend([d / filename for d in _candidate_results_dirs(settings)])

    # Fallback: directly under candidate bases.
    candidates.extend([b / filename for b in _candidate_base_dirs(settings)])

    return [p.resolve() for p in _dedupe_paths(candidates) if p.exists()]


def resolve_plot_data_path(settings: Dict) -> Path:
    found = find_plot_data_candidates(settings)
    if found:
        return found[0]

    target = settings.get("target_plot_data_file")
    tried = []
    if target is not None and str(target).strip() != "":
        tried.extend(str(p) for p in _dedupe_paths(
            [_as_path_or_none(target)]
            + [_candidate / _as_path_or_none(target) for _candidate in _candidate_results_dirs(settings) if _as_path_or_none(target) is not None and not _as_path_or_none(target).is_absolute()]
        ))
        raise FileNotFoundError(
            "Could not resolve target_plot_data_file. Tried:\n  " + "\n  ".join(tried)
        )

    filename = "Simulation_Plot_Data_All_Matches.csv"
    tried = [str(d / filename) for d in _candidate_results_dirs(settings)]
    raise FileNotFoundError(
        "Could not find Simulation_Plot_Data_All_Matches.csv. Tried:\n  "
        + "\n  ".join(tried)
        + "\n\nIf the output folder is outside the auto-detected roots, set code_root_override or target_plot_data_file in Section 1."
    )


def resolve_results_dir_path(settings: Dict, plot_data_path: Path) -> Path:
    """Return the folder that contains the resolved plotting table."""
    return Path(plot_data_path).resolve().parent

## 5. Data preparation and filtering helpers


In [ ]:
def choose_series(df: pd.DataFrame, column_candidates) -> pd.Series:
    for col in column_candidates:
        if col in df.columns:
            return pd.to_numeric(df[col], errors="coerce")
    return pd.Series(np.nan, index=df.index, dtype=float)


def signed_index(left, right, use_abs_denominator: bool = False) -> pd.Series:
    left = pd.to_numeric(left, errors="coerce")
    right = pd.to_numeric(right, errors="coerce")
    denominator = left.abs() + right.abs() if use_abs_denominator else left + right
    out = pd.Series(np.nan, index=left.index, dtype=float)
    valid = left.notna() & right.notna() & denominator.notna() & denominator.ne(0)
    out.loc[valid] = (left.loc[valid] - right.loc[valid]) / denominator.loc[valid]
    return out


def _series_or_default(df: pd.DataFrame, column: str, default: str) -> pd.Series:
    if column in df.columns:
        return df[column]
    return pd.Series(default, index=df.index)


def build_plot_fields(plot_df: pd.DataFrame, settings: Dict) -> pd.DataFrame:
    df = plot_df.copy()

    corr_source = str(settings["correlation_source"]).strip().lower()
    level_source = str(settings["level_source"]).strip().lower()

    corr_map = {
        "ref": ("shape_corr_ref", "basis_corr_ref"),
        "original": ("shape_corr_original", "basis_corr_original"),
        "simulated": ("shape_corr_simulated", "basis_corr_simulated"),
        "target": ("shape_corr_target", "basis_corr_target"),
    }
    level_map = {
        "ref": (
            "generation_mean_ref", "demand_mean_ref", "seller_lmp_mean_ref", "buyer_lmp_out_mean_ref",
            "generation_median_ref", "demand_median_ref", "seller_lmp_median_ref", "buyer_lmp_out_median_ref",
        ),
        "original": (
            "generation_original_mean", "demand_original_mean", "seller_lmp_original_mean", "buyer_lmp_out_original_mean",
            "generation_original_median", "demand_original_median", "seller_lmp_original_median", "buyer_lmp_out_original_median",
        ),
        "simulated": (
            "generation_simulated_mean", "demand_simulated_mean", "seller_lmp_simulated_mean", "buyer_lmp_out_simulated_mean",
            "generation_simulated_median", "demand_simulated_median", "seller_lmp_simulated_median", "buyer_lmp_out_simulated_median",
        ),
    }

    shape_col, basis_col = corr_map.get(corr_source, corr_map["simulated"])
    (
        g_mean_col, d_mean_col, seller_price_mean_col, buyer_price_mean_col,
        g_median_col, d_median_col, seller_price_median_col, buyer_price_median_col,
    ) = level_map.get(level_source, level_map["simulated"])

    df["shape_corr_plot"] = choose_series(df, [shape_col, "shape_corr_simulated", "shape_corr_ref", "shape_corr_original"])
    df["basis_corr_plot"] = choose_series(df, [basis_col, "basis_corr_simulated", "basis_corr_ref", "basis_corr_original"])

    df["seller_volume_mean_used"] = choose_series(df, [g_mean_col])
    df["buyer_volume_mean_used"] = choose_series(df, [d_mean_col])
    df["seller_price_mean_used"] = choose_series(df, [seller_price_mean_col])
    df["buyer_price_mean_used"] = choose_series(df, [buyer_price_mean_col])

    df["seller_volume_median_used"] = choose_series(df, [g_median_col])
    df["buyer_volume_median_used"] = choose_series(df, [d_median_col])
    df["seller_price_median_used"] = choose_series(df, [seller_price_median_col])
    df["buyer_price_median_used"] = choose_series(df, [buyer_price_median_col])

    df["vol_mismatch_mean_plot"] = signed_index(
        df["seller_volume_mean_used"],
        df["buyer_volume_mean_used"],
        use_abs_denominator=False,
    )
    df["price_spread_mean_plot"] = signed_index(
        df["seller_price_mean_used"],
        df["buyer_price_mean_used"],
        use_abs_denominator=True,
    )
    df["vol_mismatch_median_plot"] = signed_index(
        df["seller_volume_median_used"],
        df["buyer_volume_median_used"],
        use_abs_denominator=False,
    )
    df["price_spread_median_plot"] = signed_index(
        df["seller_price_median_used"],
        df["buyer_price_median_used"],
        use_abs_denominator=True,
    )

    ppa_raw = _series_or_default(df, "ppa_type", "Unknown").fillna("Unknown").astype(str)
    profile_raw = _series_or_default(df, "profile_type", "Unknown").fillna("Unknown").astype(str)
    df["selected_profile_for_switch"] = np.where(
        ppa_raw.eq("No Contract"),
        "No Contract",
        profile_raw,
    )

    return df


def _as_str_set(value) -> Optional[set[str]]:
    if value is None:
        return None
    if isinstance(value, str):
        raw_values = [value]
    else:
        raw_values = list(value)
    clean = {str(x).strip() for x in raw_values if x is not None and str(x).strip() != ""}
    return clean or None


def _as_float_list(value) -> Optional[list[float]]:
    if value is None:
        return None
    if isinstance(value, (int, float, np.integer, np.floating)):
        raw_values = [value]
    else:
        raw_values = list(value)
    out = []
    for x in raw_values:
        if x is None or str(x).strip() == "":
            continue
        out.append(float(x))
    return out or None


def _numeric_isin(series: pd.Series, allowed_values, atol: float = 1e-9) -> pd.Series:
    allowed = _as_float_list(allowed_values)
    if allowed is None:
        return pd.Series(True, index=series.index)
    numeric = pd.to_numeric(series, errors="coerce")
    keep = pd.Series(False, index=series.index)
    for value in allowed:
        keep = keep | np.isclose(numeric, value, atol=atol, equal_nan=False)
    return keep


def _is_baseline(df: pd.DataFrame) -> pd.Series:
    return df["scenario_type"].fillna("").astype(str).eq("baseline")


def filter_for_plot_content(plot_df: pd.DataFrame, settings: Dict) -> pd.DataFrame:
    """Apply all Section 1 content controls to the point-level plotting table."""
    family = str(settings["mutation_family_to_plot"])
    include_baseline = bool(settings.get("include_baseline_panel", True))

    baseline_mask_all = _is_baseline(plot_df)
    family_mask = plot_df["mutation_family"].fillna("").astype(str).eq(family)
    df = plot_df.loc[family_mask | (include_baseline & baseline_mask_all)].copy()

    baseline_mask = _is_baseline(df)

    scenario_names = _as_str_set(settings.get("filter_scenario_names"))
    if scenario_names is not None:
        df = df.loc[df["scenario_name"].astype(str).isin(scenario_names)].copy()
        baseline_mask = _is_baseline(df)

    level_labels = _as_str_set(settings.get("filter_mutation_level_labels"))
    if level_labels is not None:
        keep_levels = baseline_mask | df["mutation_level_label"].fillna("").astype(str).isin(level_labels)
        df = df.loc[keep_levels].copy()
        baseline_mask = _is_baseline(df)

    target_deltas = settings.get("filter_mutation_target_deltas")
    if target_deltas is not None:
        keep_deltas = baseline_mask | _numeric_isin(df["target_delta"], target_deltas)
        df = df.loc[keep_deltas].copy()
        baseline_mask = _is_baseline(df)

    target_physical_shifts = settings.get("filter_target_physical_shifts")
    if target_physical_shifts is not None and "target_physical_shift" in df.columns:
        keep_physical = baseline_mask | _numeric_isin(df["target_physical_shift"], target_physical_shifts)
        df = df.loc[keep_physical].copy()
        baseline_mask = _is_baseline(df)

    if settings.get("baseline_profile_filter") is not None:
        baseline_profile = str(settings["baseline_profile_filter"])
        baseline_rows_for_match_filter = plot_df.loc[baseline_mask_all].copy()
        keep_match_ids = baseline_rows_for_match_filter.loc[
            baseline_rows_for_match_filter["selected_profile_for_switch"].astype(str).eq(baseline_profile),
            "match_id",
        ].dropna().astype(int).tolist()
        df = df.loc[pd.to_numeric(df["match_id"], errors="coerce").isin(keep_match_ids)].copy()

    current_profiles = _as_str_set(settings.get("filter_current_profiles"))
    if current_profiles is not None:
        df = df.loc[df["selected_profile_for_switch"].astype(str).isin(current_profiles)].copy()

    ppa_types = _as_str_set(settings.get("filter_ppa_types"))
    if ppa_types is not None:
        df = df.loc[df["ppa_type"].fillna("Unknown").astype(str).isin(ppa_types)].copy()

    match_ids = settings.get("filter_match_ids")
    if match_ids is not None:
        allowed_ids = {int(x) for x in match_ids}
        df = df.loc[pd.to_numeric(df["match_id"], errors="coerce").isin(allowed_ids)].copy()

    return df.sort_values(["scenario_order", "match_id"]).reset_index(drop=True)




def _first_existing_column(df: pd.DataFrame, candidates: list[str]) -> Optional[str]:
    for col in candidates:
        if col in df.columns:
            return col
    return None


def _clean_choice_text(series: pd.Series, default: str = "Unknown") -> pd.Series:
    clean = series.fillna(default).astype(str).str.strip()
    return clean.mask(clean.isin(["", "nan", "None", "<NA>"]), default)


def build_choice_export_tables(plot_df: pd.DataFrame, settings: Dict) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Build compact exports that exactly follow the filtered plotting sample.

    Sheet/CSV 1: one row per plotted match-scenario observation, retaining only
    scenario identifiers plus PPA choice, profile, volume, and strike price.

    Sheet 2: scenario-level counts by combined PPA choice, including No Contract
    when it appears.
    """
    df = plot_df.copy()

    # Prefer the already selected/verified columns used by the plotting table.
    volume_col = _first_existing_column(df, [
        "volume_mw",
        "formula_selected_volume_mw",
        "best_ppa_volume_mw",
        "formula_best_ppa_volume_mw",
    ])
    price_col = _first_existing_column(df, [
        "strike_price_mwh",
        "formula_selected_strike_price_mwh",
        "best_ppa_strike_price_mwh",
        "formula_best_ppa_strike_price_mwh",
    ])

    ppa_type = _clean_choice_text(
        df["ppa_type"] if "ppa_type" in df.columns else pd.Series("Unknown", index=df.index),
        default="Unknown",
    )
    profile_type = _clean_choice_text(
        df["profile_type"] if "profile_type" in df.columns else pd.Series("Unknown", index=df.index),
        default="Unknown",
    )
    selected_profile = _clean_choice_text(
        df["selected_profile_for_switch"] if "selected_profile_for_switch" in df.columns else profile_type,
        default="Unknown",
    )

    no_contract = ppa_type.eq("No Contract") | selected_profile.eq("No Contract")
    combined_choice = np.where(
        no_contract,
        "No Contract",
        (ppa_type + " - " + profile_type).str.replace("Unknown - Unknown", "Unknown", regex=False),
    )

    scenario_cols = [
        c for c in [
            "scenario_order",
            "scenario_name",
            "scenario_type",
            "mutation_family",
            "mutation_level_label",
            "target_delta",
            "target_physical_shift",
            "calibrated_latent_delta",
        ]
        if c in df.columns
    ]

    choice_by_match = pd.DataFrame(index=df.index)
    for col in scenario_cols:
        choice_by_match[col] = df[col]
    choice_by_match["match_id"] = pd.to_numeric(df["match_id"], errors="coerce").astype("Int64") if "match_id" in df.columns else pd.Series(pd.NA, index=df.index, dtype="Int64")
    choice_by_match["ppa_choice"] = combined_choice
    choice_by_match["ppa_type"] = ppa_type
    choice_by_match["profile_type"] = profile_type
    choice_by_match["selected_profile_for_switch"] = selected_profile
    choice_by_match["volume_mw"] = pd.to_numeric(df[volume_col], errors="coerce") if volume_col is not None else np.nan
    choice_by_match["strike_price_mwh"] = pd.to_numeric(df[price_col], errors="coerce") if price_col is not None else np.nan

    sort_cols = [c for c in ["scenario_order", "match_id"] if c in choice_by_match.columns]
    if sort_cols:
        choice_by_match = choice_by_match.sort_values(sort_cols).reset_index(drop=True)
    else:
        choice_by_match = choice_by_match.reset_index(drop=True)

    # Count each PPA-choice category within each scenario. This is long-form so
    # zero/rare categories do not create unstable column layouts across runs.
    count_group_cols = scenario_cols + ["ppa_choice", "ppa_type", "selected_profile_for_switch"]
    choice_counts = (
        choice_by_match
        .groupby(count_group_cols, dropna=False)
        .agg(n_matches=("match_id", "nunique"))
        .reset_index()
    )
    scenario_totals = (
        choice_by_match
        .groupby(scenario_cols, dropna=False)
        .agg(total_matches=("match_id", "nunique"))
        .reset_index()
    )
    choice_counts = choice_counts.merge(scenario_totals, on=scenario_cols, how="left")
    choice_counts["share"] = np.where(
        choice_counts["total_matches"].gt(0),
        choice_counts["n_matches"] / choice_counts["total_matches"],
        np.nan,
    )
    choice_counts = choice_counts.sort_values(
        [c for c in ["scenario_order", "ppa_choice", "ppa_type", "selected_profile_for_switch"] if c in choice_counts.columns]
    ).reset_index(drop=True)

    return choice_by_match, choice_counts


def export_choice_tables(plot_df: pd.DataFrame, settings: Dict, save_dir: Path) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Write compact CSV and optional XLSX workbook matching the plotted rows."""
    choice_by_match, choice_counts = build_choice_export_tables(plot_df, settings)
    decimals = int(settings.get("plot_export_decimal_places", 2))
    mutation_family = settings["mutation_family_to_plot"]

    csv_path = save_dir / settings.get(
        "filtered_plot_table_name_template",
        "Filtered_Mutation_PPA_Choices__{mutation_family}.csv",
    ).format(mutation_family=mutation_family)
    choice_by_match.round(decimals).to_csv(
        csv_path,
        index=False,
        float_format=f"%.{decimals}f",
    )
    print(f"Compact choice table exported to: {csv_path}")

    if settings.get("export_choice_summary_workbook", True):
        workbook_path = save_dir / settings.get(
            "choice_summary_workbook_name_template",
            "Filtered_Mutation_PPA_Choices__{mutation_family}.xlsx",
        ).format(mutation_family=mutation_family)
        try:
            with pd.ExcelWriter(workbook_path, engine="openpyxl") as writer:
                choice_by_match.round(decimals).to_excel(writer, sheet_name="choice_by_match", index=False)
                choice_counts.round({"share": 4}).to_excel(writer, sheet_name="choice_counts", index=False)
            print(f"Choice summary workbook exported to: {workbook_path}")
        except Exception as exc:
            print(f"Could not export XLSX workbook; CSV was still written. Error: {exc}")

    return choice_by_match, choice_counts

def filter_switch_table_for_plot_content(sw: pd.DataFrame, settings: Dict) -> pd.DataFrame:
    """Apply Section 1 controls that are compatible with the aggregate switch matrix."""
    family = str(settings["mutation_family_to_plot"])
    sw = sw.loc[sw["mutation_family"].astype(str).eq(family)].copy()

    scenario_names = _as_str_set(settings.get("filter_scenario_names"))
    if scenario_names is not None:
        sw = sw.loc[sw["scenario_name"].astype(str).isin(scenario_names)].copy()

    level_labels = _as_str_set(settings.get("filter_mutation_level_labels"))
    if level_labels is not None and "mutation_level_label" in sw.columns:
        sw = sw.loc[sw["mutation_level_label"].fillna("").astype(str).isin(level_labels)].copy()

    target_deltas = settings.get("filter_mutation_target_deltas")
    if target_deltas is not None and "target_delta" in sw.columns:
        sw = sw.loc[_numeric_isin(sw["target_delta"], target_deltas)].copy()

    target_physical_shifts = settings.get("filter_target_physical_shifts")
    if target_physical_shifts is not None and "target_physical_shift" in sw.columns:
        sw = sw.loc[_numeric_isin(sw["target_physical_shift"], target_physical_shifts)].copy()

    if settings.get("baseline_profile_filter") is not None:
        sw = sw.loc[
            sw["baseline_profile_for_switch"].astype(str).eq(str(settings["baseline_profile_filter"]))
        ].copy()

    current_profiles = _as_str_set(settings.get("filter_current_profiles"))
    if current_profiles is not None:
        sw = sw.loc[sw["selected_profile_for_switch"].astype(str).isin(current_profiles)].copy()

    return sw


## 6. Plotting helpers


In [ ]:
def scenario_panel_title(row: pd.Series) -> str:
    if str(row.get("scenario_type", "")) == "baseline":
        return "Baseline"

    physical_shift = pd.to_numeric(row.get("target_physical_shift", np.nan), errors="coerce")
    latent_delta = pd.to_numeric(row.get("target_delta", np.nan), errors="coerce")
    label = str(row.get("mutation_level_label", "")).strip()

    if pd.notna(physical_shift):
        return f"Target physical $\\Delta\\rho$ = {physical_shift:+.2f}"
    if pd.notna(latent_delta):
        return f"Latent $\\Delta\\rho$ = {latent_delta:+.2f}"
    return label or str(row.get("scenario_name", ""))


def scatter_columns_and_labels(scatter_map: str) -> Tuple[str, str, str, str]:
    mode = str(scatter_map).strip().lower()
    if mode == "mean":
        return (
            "vol_mismatch_mean_plot",
            "price_spread_mean_plot",
            "Volume mismatch index (expectation)",
            "Price spread index (expectation)",
        )
    if mode == "median":
        return (
            "vol_mismatch_median_plot",
            "price_spread_median_plot",
            "Volume mismatch index (median)",
            "Price spread index (median)",
        )
    return (
        "shape_corr_plot",
        "basis_corr_plot",
        "Profile Shape Correlation",
        "Nodal Price Correlation",
    )


def _ordered_legend_labels(present_labels, preferred_order: Dict) -> list:
    present = {
        str(label).strip()
        for label in present_labels
        if str(label).strip() not in {"", "nan", "<NA>", "None"}
    }
    ordered = [label for label in preferred_order.keys() if label in present]
    extras = sorted(label for label in present if label not in preferred_order)
    return ordered + extras


def _common_limits(series: pd.Series, manual_lim=None, pad_frac: float = 0.06):
    if manual_lim is not None:
        return manual_lim
    x = pd.to_numeric(series, errors="coerce").dropna()
    if x.empty:
        return None
    lo, hi = float(x.min()), float(x.max())
    lo = min(lo, 0.0)
    hi = max(hi, 0.0)
    if hi <= lo:
        return (lo - 1.0, hi + 1.0)
    pad = pad_frac * (hi - lo)
    return (lo - pad, hi + pad)


def _build_profile_legend_handles(profile_series: pd.Series, color_map: Dict, marker_map: Dict, settings: Dict):
    unknown_color = color_map.get("Unknown", "#d62728")
    clean = profile_series.fillna("Unknown").astype(str).str.strip()
    clean = clean.mask(clean.eq(""), "Unknown")
    counts = clean.value_counts().to_dict()
    labels = _ordered_legend_labels(counts.keys(), color_map)
    handles = []
    for label in labels:
        count = int(counts.get(label, 0))
        if count <= 0:
            continue
        display_label = f"{label} ({count})" if settings.get("profile_legend_count_label", True) else label
        handles.append(
            Line2D(
                [0], [0],
                marker=(marker_map.get("No Contract", "X") if label == "No Contract" else "o"),
                linestyle="",
                markersize=settings["legend_marker_size"],
                markerfacecolor=color_map.get(label, unknown_color),
                markeredgecolor=settings["edge_color"],
                markeredgewidth=settings["edge_width"],
                label=display_label,
            )
        )
    return handles


def plot_choice_evolution_panels(plot_df: pd.DataFrame, settings: Dict, save_dir: Path) -> None:
    x_col, y_col, x_label, y_label = scatter_columns_and_labels(settings["scatter_map"])
    panel_cols = ["scenario_name", "scenario_type", "scenario_order", "mutation_level_label", "target_delta"]
    for extra_col in ["target_physical_shift", "calibrated_latent_delta"]:
        if extra_col in plot_df.columns and extra_col not in panel_cols:
            panel_cols.append(extra_col)

    panel_meta = (
        plot_df[panel_cols]
        .drop_duplicates()
        .sort_values("scenario_order")
        .reset_index(drop=True)
    )

    if panel_meta.empty:
        raise ValueError("No scenarios remain after filtering.")

    n_panels = len(panel_meta)
    n_cols = 2 if n_panels > 1 else 1
    n_rows = int(np.ceil(n_panels / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=settings["figsize"], squeeze=False)
    axes_flat = axes.ravel()

    pad_frac = float(settings.get("axis_limit_padding_fraction", 0.06))
    x_lim = _common_limits(plot_df[x_col], settings.get("x_axis_lim"), pad_frac=pad_frac)
    y_lim = _common_limits(plot_df[y_col], settings.get("y_axis_lim"), pad_frac=pad_frac)

    rng = np.random.default_rng(settings["random_seed"])
    marker_map = settings["marker_map"]
    color_map = settings["color_map"]
    unknown_color = color_map.get("Unknown", "#d62728")

    for ax_idx, meta_row in panel_meta.iterrows():
        ax = axes_flat[ax_idx]
        scenario_name = str(meta_row["scenario_name"])
        sub = plot_df.loc[plot_df["scenario_name"].astype(str).eq(scenario_name)].copy()
        sub = sub.loc[sub[x_col].notna() & sub[y_col].notna()].copy()

        x_vals = sub[x_col].to_numpy(dtype=float)
        y_vals = sub[y_col].to_numpy(dtype=float)
        if settings["use_jitter"] and len(sub):
            x_vals = x_vals + rng.normal(0.0, settings["jitter_scale_x"], size=len(sub))
            y_vals = y_vals + rng.normal(0.0, settings["jitter_scale_y"], size=len(sub))

        ppa_series = sub.get("ppa_type", pd.Series("Unknown", index=sub.index)).fillna("Unknown").astype(str).str.strip()
        profile_series = sub.get("selected_profile_for_switch", pd.Series("Unknown", index=sub.index)).fillna("Unknown").astype(str).str.strip()
        ppa_series = ppa_series.mask(ppa_series.eq(""), "Unknown")
        profile_series = profile_series.mask(profile_series.eq(""), "Unknown")

        for idx, (_, row) in enumerate(sub.iterrows()):
            ppa_type = ppa_series.iloc[idx]
            profile_type = profile_series.iloc[idx]
            ax.scatter(
                x_vals[idx],
                y_vals[idx],
                s=settings["point_size"],
                alpha=settings["point_alpha"],
                marker=marker_map.get(ppa_type, "o"),
                color=color_map.get(profile_type, unknown_color),
                edgecolors=settings["edge_color"],
                linewidths=settings["edge_width"],
            )
            if settings["show_match_id_labels"]:
                ax.text(
                    x_vals[idx],
                    y_vals[idx],
                    str(int(row["match_id"])),
                    fontsize=settings["label_font_size"],
                    alpha=settings.get("label_alpha", 0.8),
                )

        zero_line_width = settings.get("axis_zero_line_width", 1.0)
        ax.axhline(0.0, linewidth=zero_line_width)
        ax.axvline(0.0, linewidth=zero_line_width)
        if x_lim is not None:
            ax.set_xlim(x_lim)
        if y_lim is not None:
            ax.set_ylim(y_lim)
        ax.set_title(scenario_panel_title(meta_row), fontsize=settings["subplot_title_font_size"])
        ax.set_xlabel(x_label, fontsize=settings["axis_title_font_size"])
        ax.set_ylabel(y_label, fontsize=settings["axis_title_font_size"])
        ax.tick_params(axis="both", which="both", labelsize=settings["tick_font_size"])

        if settings.get("show_profile_type_legend", True) and settings.get("profile_legend_per_panel", True):
            profile_handles = _build_profile_legend_handles(profile_series, color_map, marker_map, settings)
            if profile_handles:
                ax.legend(
                    handles=profile_handles,
                    title="Profile Type",
                    loc=settings.get("profile_legend_loc", "lower left"),
                    fontsize=settings["legend_font_size"],
                    title_fontsize=settings["legend_title_font_size"],
                    frameon=True,
                )

    for ax in axes_flat[n_panels:]:
        ax.axis("off")

    fig.tight_layout()

    if settings["save_fig"]:
        save_dir.mkdir(parents=True, exist_ok=True)
        fig_name = settings["choice_fig_name_template"].format(
            mutation_family=settings["mutation_family_to_plot"],
            scatter_map=settings["scatter_map"],
        )
        fig_path = save_dir / fig_name
        fig.savefig(fig_path, dpi=300, bbox_inches="tight")
        print(f"Saved figure: {fig_path}")

    plt.show()


def plot_switch_bars(results_dir_path: Path, settings: Dict, save_dir: Path) -> None:
    unsupported_filters = [
        name for name in ["filter_match_ids", "filter_ppa_types"]
        if settings.get(name) is not None
    ]
    if unsupported_filters:
        print(
            "Skipping switch bars because the aggregate switch matrix cannot apply these point-level filters: "
            + ", ".join(unsupported_filters)
        )
        return

    switch_file = results_dir_path / "Mutation_Profile_Switch_Matrix.csv"
    if not switch_file.exists():
        print(f"Skipping switch bars; file not found: {switch_file}")
        return

    sw = read_csv_optimized(switch_file)
    family = str(settings["mutation_family_to_plot"])
    sw = filter_switch_table_for_plot_content(sw, settings)
    if sw.empty:
        print(f"Skipping switch bars; no switch rows remain for family {family!r} after filtering.")
        return

    outcome_order = ["Fix", "AsC", "AsG", "No Contract", "Unknown"]
    scenario_cols = ["scenario_name", "scenario_order", "target_delta", "mutation_level_label"]
    if "target_physical_shift" in sw.columns:
        scenario_cols.append("target_physical_shift")
    scenario_order = (
        sw[scenario_cols]
        .drop_duplicates()
        .sort_values("scenario_order")
        .reset_index(drop=True)
    )

    baseline_profiles = sorted(sw["baseline_profile_for_switch"].dropna().astype(str).unique().tolist())
    n_profiles = len(baseline_profiles)
    if n_profiles == 0:
        return

    width = max(settings["bar_figsize_min_width"], settings["bar_figsize_per_profile"] * n_profiles)
    fig, axes = plt.subplots(
        1,
        n_profiles,
        figsize=(width, settings["bar_figsize_height"]),
        squeeze=False,
    )
    axes_flat = axes.ravel()

    for ax_idx, base_profile in enumerate(baseline_profiles):
        ax = axes_flat[ax_idx]
        sub = sw.loc[sw["baseline_profile_for_switch"].astype(str).eq(base_profile)].copy()
        x = np.arange(len(scenario_order))
        bottoms = np.zeros(len(scenario_order))

        for outcome in outcome_order:
            vals = []
            for _, sc in scenario_order.iterrows():
                row = sub.loc[
                    sub["scenario_name"].astype(str).eq(str(sc["scenario_name"]))
                    & sub["selected_profile_for_switch"].astype(str).eq(outcome)
                ]
                vals.append(float(row["share_within_baseline_profile"].sum()) if not row.empty else 0.0)
            if max(vals) > 0:
                ax.bar(x, vals, bottom=bottoms, label=outcome)
                bottoms += np.asarray(vals)

        labels = []
        for _, sc in scenario_order.iterrows():
            physical_shift = pd.to_numeric(sc.get("target_physical_shift", np.nan), errors="coerce")
            latent_delta = pd.to_numeric(sc.get("target_delta", np.nan), errors="coerce")
            if pd.notna(physical_shift):
                labels.append(f"{physical_shift:+.2f}")
            elif pd.notna(latent_delta):
                labels.append(f"{latent_delta:+.2f}")
            else:
                labels.append(str(sc.get("mutation_level_label", "")))

        ax.set_xticks(x)
        ax.set_xticklabels(labels, fontsize=settings["tick_font_size"])
        ax.set_ylim(settings.get("bar_ylim", (0, 1.02)))
        ax.set_title(f"Baseline {base_profile}", fontsize=settings["subplot_title_font_size"])
        ax.set_xlabel(settings.get("bar_xlabel", "Target physical $\\Delta\\rho$"), fontsize=settings["axis_title_font_size"])
        ax.set_ylabel(settings.get("bar_ylabel", "Share"), fontsize=settings["axis_title_font_size"])
        ax.tick_params(axis="y", labelsize=settings["tick_font_size"])

    axes_flat[0].legend(
        fontsize=settings["legend_font_size"],
        title=settings.get("bar_legend_title", "Mutated profile"),
        title_fontsize=settings["legend_title_font_size"],
    )
    fig.tight_layout()

    if settings["save_fig"]:
        save_dir.mkdir(parents=True, exist_ok=True)
        fig_name = settings["switch_fig_name_template"].format(mutation_family=family)
        fig_path = save_dir / fig_name
        fig.savefig(fig_path, dpi=300, bbox_inches="tight")
        print(f"Saved switch figure: {fig_path}")

    plt.show()


## 7. Execute

Run this section after editing Sections 1 and 2.


In [ ]:
plot_data_path = resolve_plot_data_path(SETTINGS)

# Keep all downstream outputs anchored to the directory where the plotting CSV
# was actually found. This avoids mismatches when the notebook is executed from
# the root folder or from the simulation_mutation subfolder.
results_dir_path = resolve_results_dir_path(SETTINGS, plot_data_path)
save_dir = (results_dir_path / SETTINGS["save_dir_name"]).resolve()
save_dir.mkdir(parents=True, exist_ok=True)

raw_df = read_csv_optimized(plot_data_path)
plot_df_full = build_plot_fields(raw_df, SETTINGS)
plot_df_plot = filter_for_plot_content(plot_df_full, SETTINGS)

print("Resolved paths:")
print("  Notebook cwd    :", NOTEBOOK_CWD)
print("  Results dir     :", results_dir_path)
print("  Plot-data table :", plot_data_path)
print("  Save directory  :", save_dir)
all_candidates = find_plot_data_candidates(SETTINGS)
if len(all_candidates) > 1:
    print("  Other plot-data candidates found:")
    for candidate in all_candidates[1:8]:
        print("    -", candidate)
print()
print("Mutation family:", SETTINGS["mutation_family_to_plot"])
print("Rows before filtering:", len(plot_df_full))
print("Rows after filtering :", len(plot_df_plot))
if SETTINGS.get("baseline_profile_filter") is not None:
    print("Baseline profile filter:", SETTINGS["baseline_profile_filter"])

if plot_df_plot.empty:
    raise ValueError("No rows remain after Section 1 content filters. Relax one or more filters and rerun.")

choice_by_match = None
choice_counts = None
if SETTINGS["export_filtered_plot_table"]:
    choice_by_match, choice_counts = export_choice_tables(plot_df_plot, SETTINGS, save_dir)

print("\nScenario counts:")
print(
    plot_df_plot[["scenario_name", "scenario_order"]]
    .drop_duplicates()
    .sort_values("scenario_order")
    .to_string(index=False)
)

print("\nProfile counts by scenario:")
print(pd.crosstab(plot_df_plot["scenario_name"], plot_df_plot["selected_profile_for_switch"]).to_string())

if choice_by_match is not None:
    print("\nPPA-choice counts by scenario:")
    print(pd.crosstab(choice_by_match["scenario_name"], choice_by_match["ppa_choice"]).to_string())

if SETTINGS["plot_choice_evolution_figure"]:
    plot_choice_evolution_panels(plot_df_plot, SETTINGS, save_dir)

if SETTINGS["plot_switch_bars_figure"]:
    plot_switch_bars(results_dir_path, SETTINGS, save_dir)
